<a href="https://colab.research.google.com/github/komazawa-deep-learning/komazawa-deep-learning.github.io/blob/master/2026notebooks/20261023JGLUE_BERT_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# JGLUE で Transformer（BERT）を動かしてみる
- データ: [yahoojapan/JGLUE](https://github.com/yahoojapan/JGLUE)（CC BY-SA 4.0）。ここでは JNLI（文ペアの含意関係）と JCommonsenseQA を**眺める**のと、JNLI の小さな部分集合で**ごく短い微調整**を回すだけ。
- モデル: `tohoku-nlp/bert-base-japanese-v3`（Apache-2.0, 12層・768次元・語彙32,768）。
- **精度の評価・比較は目的外**。微調整は数千件・1エポックなので、出る数値は「動いた」ことの確認にしかならない。
- 注意: 作成環境から Hugging Face に接続できず、**モデルの読み込み部分は実機で未検証**（データ読み込みと学習ループは、ランダム初期化の小型BERTで動作確認済み）。
- MARC-ja は Amazon が配布を停止したため JGLUE から利用不可。JSQuAD は本ノートでは扱わない。

## 0. インストール
`transformers<5` に固定する（日本語BERTトークナイザーの読み込みを確実にするため。5系での動作は未確認）。入れ替えたら**ランタイムを再起動**。

In [1]:
import transformers
transformers.__version__

'4.57.6'

In [2]:
import sys, subprocess
def pip(*a): subprocess.run([sys.executable,"-m","pip","install","-q",*a],check=True)
pip("transformers>=4.44,<5","accelerate","fugashi","unidic-lite","pandas","matplotlib")
import transformers; print("transformers", transformers.__version__)

transformers 4.57.6


In [3]:
import os, random, time
import numpy as np, pandas as pd, torch
SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    DEV="cuda"
elif torch.backends.mps.is_available():
    DEV="mps"
else: DEV="cpu"
print("device:", DEV)

device: cpu


## 1. JGLUE を取得して中身を見る
各ファイルは拡張子が `.json` だが、**1行1レコードの JSONL**（`lines=True` で読む）。dev に相当するのは `valid-v1.3.json`。

In [5]:
import IPython
isColab = 'google.colab' in str(IPython.get_ipython())

if not os.path.isdir("JGLUE"):
    subprocess.run(["git","clone","--depth","1","https://github.com/yahoojapan/JGLUE"],check=True)
D="JGLUE/datasets"
if not isColab:
    D = os.path.join(HOME, 'study/2026yahoojapan_JGLUE.git')

rd=lambda t,s: pd.read_json(f"{D}/{t}-v1.3/{s}-v1.3.json", lines=True)
jnli={s:rd("jnli",s) for s in ["train","valid","test"]}
jsts={s:rd("jsts",s) for s in ["train","valid","test"]}
jcqa={s:rd("jcommonsenseqa",s) for s in ["train","valid","test"]}
for name,d in [("JNLI",jnli),("JSTS",jsts),("JCommonsenseQA",jcqa)]:
    print(name, {k:len(v) for k,v in d.items()})

JNLI {'train': 20073, 'valid': 2434, 'test': 2508}
JSTS {'train': 12451, 'valid': 1457, 'test': 1589}
JCommonsenseQA {'train': 8939, 'valid': 1119, 'test': 1118}


In [ ]:
pd.set_option("display.max_colwidth",60, "display.width",200)
print("--- JNLI（2文の関係: entailment / contradiction / neutral）")
display(jnli["train"].sample(5, random_state=SEED)[["sentence1","sentence2","label"]])
print("JNLI ラベル分布(train):"); print(jnli["train"].label.value_counts())
print("--- JSTS（類似度 0〜5）")
display(jsts["train"].sample(5, random_state=SEED)[["sentence1","sentence2","label"]])
print("--- JCommonsenseQA（5択。label が正解の番号）")
display(jcqa["train"].sample(3, random_state=SEED))

## 2. トークナイザー: 文が BERT にどう入るか

In [ ]:
from transformers import AutoTokenizer
MID="tohoku-nlp/bert-base-japanese-v3"
tok=AutoTokenizer.from_pretrained(MID)
s1,s2=jnli["train"].sentence1[0], jnli["train"].sentence2[0]
enc=tok(s1,s2)
print(s1,"|",s2)
print(tok.convert_ids_to_tokens(enc["input_ids"]))
print("input_ids     :",enc["input_ids"])
print("token_type_ids:",enc["token_type_ids"], " ← 文1=0, 文2=1")

## 3. 事前学習済み BERT そのまま: [MASK] の穴埋め（微調整なし）

In [ ]:
from transformers import AutoModelForMaskedLM
mlm=AutoModelForMaskedLM.from_pretrained(MID).to(DEV).eval()
def fill(text, k=5):
    x=tok(text, return_tensors="pt").to(DEV)
    pos=(x["input_ids"][0]==tok.mask_token_id).nonzero().item()
    with torch.no_grad(): lg=mlm(**x).logits[0,pos]
    p=lg.softmax(-1); top=p.topk(k)
    return [(tok.convert_ids_to_tokens(int(i)), round(float(v),3)) for v,i in zip(top.values,top.indices)]
for t in ["今日は天気が良いので、公園を[MASK]しました。",
          "東京は日本の[MASK]です。",
          "彼女はコーヒーに[MASK]を入れて飲んだ。"]:
    print(t); print("  ",fill(t))

## 4. Attention を見る（どの語がどの語を見ているか）
1つの層・1つのヘッドを描くだけ。**ヘッドの役割を解釈するのは慎重に**（見た目の「それらしさ」と因果は別）。

In [ ]:
import matplotlib.pyplot as plt, subprocess
if "google.colab" in sys.modules:
    subprocess.run("apt-get -qq install -y fonts-ipafont-gothic >/dev/null",shell=True)
from matplotlib import font_manager as fm
for f in fm.findSystemFonts():
    if any(k in f for k in ["ipag","IPAGothic","ipafont","NotoSansCJK","Hiragino","AppleGothic"]):
        fm.fontManager.addfont(f)
cand=[n for n in ["IPAGothic","IPAexGothic","Noto Sans CJK JP","Hiragino Sans","Hiragino Maru Gothic Pro"] if n in {x.name for x in fm.fontManager.ttflist}]
if cand: plt.rcParams["font.family"]=cand[0]
else: print("日本語フォントが見つからない: 文字が□になる場合は上の apt/フォント設定を確認")

from transformers import AutoModel
enc_model=AutoModel.from_pretrained(MID, attn_implementation="eager").to(DEV).eval()   # eager でないと attention を返せない
text="犬が公園で子どもと遊んでいます。"
x=tok(text, return_tensors="pt").to(DEV)
with torch.no_grad(): out=enc_model(**x, output_attentions=True)
toks=tok.convert_ids_to_tokens(x["input_ids"][0])
LAYER,HEAD=8,0
A=out.attentions[LAYER][0,HEAD].float().cpu().numpy()
plt.figure(figsize=(5,5)); plt.imshow(A,cmap="viridis")
plt.xticks(range(len(toks)),toks,rotation=90); plt.yticks(range(len(toks)),toks)
plt.title(f"layer {LAYER}, head {HEAD}（行=見る側, 列=見られる側）"); plt.colorbar(shrink=.7); plt.tight_layout(); plt.show()

## 5. JNLI でごく短い微調整（動作確認用）
- 学習 N_TRAIN 件・1 エポック。**これは学習の様子を見せるための設定で、性能を主張するものではない**。
- 3値分類。ラベルは `entailment=0, contradiction=1, neutral=2` と本ノートで決めた対応。
- 参考: JNLI train の多数派クラス（neutral）が 55.8%（11,193/20,073）。多数派だけを答える分類器でもこの程度の正解率になるので、**数値を読むときの基準**にすること。

In [ ]:
from transformers import AutoModelForSequenceClassification
LABELS=["entailment","contradiction","neutral"]; L2I={l:i for i,l in enumerate(LABELS)}
N_TRAIN, N_EVAL = 3000, 500
BATCH, EPOCHS, LR, MAXLEN = 32, 1, 3e-5, 64
tr=jnli["train"].sample(N_TRAIN, random_state=SEED).reset_index(drop=True)
va=jnli["valid"].sample(N_EVAL,  random_state=SEED).reset_index(drop=True)

def batches(df, shuffle):
    idx=np.random.permutation(len(df)) if shuffle else np.arange(len(df))
    for i in range(0,len(df),BATCH):
        b=df.iloc[idx[i:i+BATCH]]
        x=tok(list(b.sentence1), list(b.sentence2), truncation=True, max_length=MAXLEN,
              padding=True, return_tensors="pt")
        y=torch.tensor([L2I[l] for l in b.label])
        yield {k:v.to(DEV) for k,v in x.items()}, y.to(DEV)

clf=AutoModelForSequenceClassification.from_pretrained(MID, num_labels=3).to(DEV)
opt=torch.optim.AdamW(clf.parameters(), lr=LR, weight_decay=0.01)
steps=EPOCHS*((N_TRAIN+BATCH-1)//BATCH)
sched=torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min((s+1)/(0.1*steps), max(0.0,(steps-s)/(0.9*steps))))

def evaluate():
    clf.eval(); P=[]; Y=[]
    with torch.no_grad():
        for x,y in batches(va, False):
            P.append(clf(**x).logits.argmax(-1).cpu()); Y.append(y.cpu())
    P=torch.cat(P); Y=torch.cat(Y); return (P==Y).float().mean().item(), P

acc0,_=evaluate(); print(f"微調整前（分類層はランダム）: 正解率 {acc0:.3f}")
losses=[]; t0=time.time()
for ep in range(EPOCHS):
    clf.train()
    for x,y in batches(tr, True):
        loss=clf(**x, labels=y).loss
        loss.backward(); torch.nn.utils.clip_grad_norm_(clf.parameters(),1.0)
        opt.step(); sched.step(); opt.zero_grad(); losses.append(loss.item())
print(f"学習 {time.time()-t0:.0f} 秒")
acc1,pred=evaluate(); print(f"微調整後: 正解率 {acc1:.3f}  （評価 {N_EVAL} 件。多数派ベースライン ≒ {(va.label=='neutral').mean():.3f}）")
plt.figure(figsize=(5,3)); plt.plot(losses); plt.xlabel("step"); plt.ylabel("train loss"); plt.tight_layout(); plt.show()

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at tohoku-nlp/bert-base-japanese-v3 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.


In [ ]:
va2=va.copy(); va2["pred"]=[LABELS[i] for i in pred.tolist()]
print("混同の様子:"); display(pd.crosstab(va2.label, va2.pred))
print("先頭10件の予測例（選別していない。食い違いも含む）")
display(va2.head(10)[["sentence1","sentence2","label","pred"]])